# Generate and Analyze Geometry Dataset

Generate a synthetic dataset for the PhaseGridUNet model.

**Workflow**

1. Set `PRESET = "smoke"` and run all sections to verify dataset generation.
2. Set `PRESET = "development"` and run all sections to generate the development dataset.
3. Review the sampling distributions, generation performance and dataset previews before training.

To inspect an existing dataset, set `DATASET_NAME` accordingly and skip dataset generation.


## 1. Setup

Use the corresponding experiment's Python environment as the notebook kernel.


In [ ]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import grid_geometry
from grid_geometry.dataset.generator import GENERATOR_REVISION, generate_dataset
from grid_geometry.dataset.sampling_config import GenerationConfig
from grid_geometry.dataset.validation import validate_saved_dataset

DATA_DIR = Path(grid_geometry.__file__).resolve().parents[2] / "datasets"
from grid_geometry.dataset.visualize import visualize_samples
from grid_geometry.notebook_display import show_table


## 2. Configuration

Set `DATASET_NAME` to choose a different output folder, `SEED` to generate a different reproducible dataset, and `WORKERS` to control parallel generation.


In [ ]:
PRESET = "smoke"       # "smoke", "development"
DATASET_NAME = PRESET
SEED = 42              # Reproducible sampling seed
WORKERS = os.process_cpu_count() or 1  # Generation processes

PRESETS = {
    "smoke": {
        "train": 40,
        "val": 12,
        "test": 12,
    },
    "development": {
        "train": 4000,
        "val": 500,
        "test": 500,
    },
}
splits = ("train", "val", "test")
counts = PRESETS[PRESET]
destination = DATA_DIR / DATASET_NAME
config = GenerationConfig(**counts, seed=SEED, workers=WORKERS)
if GENERATOR_REVISION != 1:
    raise RuntimeError(f"Expected generator revision 1, found {GENERATOR_REVISION}")
print(f"Dataset destination: {destination}")
print(f"Preset: {PRESET}; splits: " + ", ".join(f"{k}={v}" for k, v in counts.items()))
print(f"Generator revision: {GENERATOR_REVISION}; seed: {SEED}; workers: {WORKERS}")


## 3. Generate dataset

Generation writes to the configured output folder and refuses to overwrite an existing dataset.


In [ ]:
if destination.exists():
    raise FileExistsError(f"Dataset destination already exists: {destination}")
generate_dataset(destination, config)
print(f"Generated dataset: {destination}")


## 4. Load saved dataset

The following analysis uses the saved dataset configuration, statistics and samples, rather than the current generation settings.


In [ ]:
metadata = validate_saved_dataset(destination)
statistics = json.loads((destination / "statistics.json").read_text())
with (destination / "manifest.jsonl").open() as manifest:
    records = [json.loads(line) for line in manifest if line.strip()]
positives = [r for r in records if r.get("board_id") is not None]
print(f"Loaded dataset: {destination}")
show_table("Saved dataset", ("Field", "Value"),
           [(key, metadata.get(key)) for key in ("experiment", "generator_revision", "status", "seed", "workers", *splits)])


## 5. Analyze sampling distributions

Compare configured, attempted and accepted distributions to identify deviations from the intended sampling policy caused by sampling variability, placement constraints and rejected attempts. Distributions are compared only where comparable observations are available; a missing series does not mean zero probability.


In [ ]:
accepted = statistics.get("accepted", {}).get("overall", {})
categorical = accepted.get("categorical", {})
numerical = accepted.get("numerical", {})
totals = accepted.get("totals", {})
attempts = statistics.get("attempts", {}).get("overall", {})

def category(key):
    return categorical.get(key, {}).get("counts", {})

def attempted(key, board=False):
    source = attempts.get("boards_by_containing_attempt_outcome" if board else "by", {})
    values = source.get(key, {})
    return {str(k): v.get("attempted", 0) for k, v in values.items()}

def bars(ax, title, configured, observed, accepted_counts):
    series = [("Configured", configured, "#648bc5"), ("Attempted", observed, "#e2ae65"),
              ("Accepted", accepted_counts, "#54a68b")]
    series = [(label, {str(k): v for k, v in values.items()}, color)
              for label, values, color in series if values and sum(values.values()) > 0]
    ax.set_title(title)
    ax.set_ylabel("Percent")
    if not series:
        ax.text(.5, .5, "Unavailable", ha="center", transform=ax.transAxes)
        return
    keys = list(dict.fromkeys(k for _, values, _ in series for k in values))
    width = .8 / len(series)
    for i, (label, values, color) in enumerate(series):
        total = sum(values.values())
        ax.bar([x + (i - (len(series)-1)/2)*width for x in range(len(keys))],
               [100 * values.get(k, 0) / total for k in keys], width, color=color, label=label)
    ax.set_xticks(range(len(keys)), keys, rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0, 100)

def histogram(ax, title, key, unit, xlabel, observations):
    ax.set(title=title, ylabel=unit, xlabel=xlabel)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    values = [v for v in observations if isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)]
    summary = numerical.get(key, {})
    expected = summary.get("count")
    if values and (expected is None or expected == len(values)):
        ax.hist(values, bins=min(20, max(5, math.ceil(math.sqrt(len(values))))), color="#54a68b", edgecolor="white")
        return
    hist = summary.get("histogram", {})
    edges, counts = hist.get("edges", []), hist.get("counts", [])
    if counts and len(edges) == len(counts) + 1:
        labels = [f"[{a:g}, {b:g}{']' if i == len(counts)-1 else ')'}" for i, (a, b) in enumerate(zip(edges, edges[1:]))]
        counts = list(counts)
        if hist.get("underflow"):
            labels.insert(0, f"< {edges[0]:g}")
            counts.insert(0, hist["underflow"])
        if hist.get("overflow"):
            labels.append(f"> {edges[-1]:g}")
            counts.append(hist["overflow"])
        ax.bar(range(len(counts)), counts, color="#54a68b", edgecolor="white")
        ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=7)
        ax.set_xlabel(xlabel + " — saved intervals")
    else:
        ax.text(.5, .5, "Unavailable", ha="center", transform=ax.transAxes)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
charts = [("Crop category", {}, {}, category("category")),
          ("Positive / negative", {"positive": 1-metadata["negative_fraction"], "negative": metadata["negative_fraction"]} if "negative_fraction" in metadata else {}, {}, {"positive": totals.get("positive_images", 0), "negative": totals.get("negative_images", 0)} if totals else {}),
          ("Source scene mode", {}, attempted("source_mode"), category("source_mode")),
          ("Game style", {name: 1 for name in metadata.get("source_policy", {}).get("game_styles_uniform", [])}, attempted("game_style", True), category("board_style")),
          ("Game zoom", metadata.get("game_zoom", {}), attempted("game_zoom", True), category("board_requested_zoom")),
          ("Desktop display scale", metadata.get("display_scale", {}), attempted("display_scale"), category("source_desktops_display_scale"))]
for ax, (title, configured, observed, counts) in zip(axes.flat, charts): bars(ax, title, configured, observed, counts)
histogram(axes.flat[6], "Effective cell pitch X", "final_pitch_x", "Positive samples", "Pitch (pixels)", [r.get("effective_pitch_x") for r in positives])
histogram(axes.flat[7], "Padding fraction", "padding_fraction", "Samples", "Fraction of image area", [r.get("padding_fraction") for r in records])
histogram(axes.flat[8], "Anisotropy ratio", "anisotropy_ratio", "Positive samples", "Scale ratio", [r.get("anisotropy_ratio") for r in positives])
handles, labels = [], []
for ax in axes.flat:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        if label not in labels: handles.append(handle); labels.append(label)
if handles: fig.legend(handles, labels, loc="upper center", ncol=len(labels))
fig.tight_layout(rect=(0, 0, 1, .96))
plt.show()


## 6. Analyze generation performance

Acceptance rate and attempts per accepted image indicate sampling efficiency; generation duration and throughput indicate computational performance.


In [ ]:
ATTEMPT_LABEL = "Source attempts per fixed objective"
performance = statistics.get("performance", {})
environment = metadata.get("provenance", {}).get("environment", {})
attempts = statistics.get("attempts", {})
overall = attempts.get("overall", {})
def number(value, fmt=""):
    return "unavailable" if value is None else format(value, fmt)
total, count = overall.get("attempted"), overall.get("accepted")
mean = total / count if count and total is not None else None
print(f"Generated images: {number(performance.get('accepted_image_count'), ',')}")
print(f"Mean attempts per accepted image: {number(mean, '.2f')}")
print(f"Generation duration: {number(performance.get('overall_generation_seconds'), '.2f')} s")
print(f"Effective images per second: {number(performance.get('overall_images_per_second'), '.2f')}")
print(f"Workers: {number(metadata.get('workers'))}")
print(f"Available CPUs: {number(environment.get('available_cpu_count'))}")
rows = [("Overall", overall)] + [(label, attempts.get("splits", {}).get(split, {}))
                               for split, label in (("train", "Train"), ("val", "Validation"), ("test", "Test"))]
def acceptance(item):
    attempted, accepted = item.get("attempted"), item.get("accepted")
    return accepted / attempted if attempted and accepted is not None else None
show_table(ATTEMPT_LABEL, ("Split", "Accepted attempts", "Rejected attempts", "Acceptance rate"),
           [(name, number(item.get("accepted"), ","), number(item.get("rejected"), ","),
             number(acceptance(item), ".2%")) for name, item in rows])
rejected = overall.get("rejected")
reasons = overall.get("rejection_reasons")
if rejected == 0:
    print("Rejection reasons: no attempts were rejected.")
elif isinstance(reasons, dict) and reasons:
    print("Rejection reasons:")
    for reason, count in sorted(reasons.items()):
        print(f"  {reason}: {number(count, ',')}")
else:
    print("Rejection reasons: unavailable.")


## 7. Visualize dataset samples

Inspect lattice alignment, directional phase targets, occupancy targets and their supervision masks in reproducible random examples and selected representative or challenging cases, including crop categories and pitch extremes.


In [ ]:
visualize_samples(records, positives, destination, seed=SEED)
